# BLB Hebrew Pronunciation Audio Downloader + Analyser + Splitter

**Cell 1** — Config  
**Cell 2** — Setup (folders, cache)  
**Cell 3** — Helpers  
**Cell 4** — Download loop (resume-safe)  
**Cell 5** — Silence analysis → CSV  
**Cell 6** — Split / reformat audio with ffmpeg  

Re-running any cell is safe — already-done work is skipped.

# Prepare and Define

## Cell 1 — Config

In [ ]:
START      = 1        # first Strong's number
END        = 100      # last Strong's number (inclusive)

MUSIC_DIR   = "music"        # raw downloaded audio
HTML_DIR    = "strongs_h"    # cached lexicon HTML pages
SPLIT_DIR   = "strongs_p"    # final split/reformatted audio

HASH_CACHE_FILE   = f"{MUSIC_DIR}{os.sep}blb_pronunc_hashes.json"
SILENCE_CSV_FILE  = f"{MUSIC_DIR}{os.sep}blb_silence_analysis.csv"

DELAY_OK    = 0.15#1.5   # seconds between successful requests
DELAY_RETRY = 30    # base backoff seconds on rate-limit / timeout
MAX_RETRIES = 5

# ffmpeg silence detection thresholds
SILENCE_DB       = -37    # dB threshold — adjust if too many/few gaps detected
SILENCE_MIN_DUR  = 0.350    # minimum gap duration in seconds to count as silence
SILENCE_SPIKES_MERGE_DURATION_MAX = 0.200 #max lenght of noise spikes that occur between two silences and will be treated as silence instead
SILENCE_TRAILING_SILENCE_MAX_DURATION_TO_BE_MERGED_WITH_PREVIOUS_CHUNK = 0.4 #max length of trailing silence to be merged with previous chhung (speech)
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ 2 excepts on 200. now what if.
SILENCE_DB       = -50 

## Cell 2 — Setup

In [ ]:
import os, json, time, re, shutil, subprocess
import urllib.request, urllib.error, urllib.parse
from pathlib import Path
import pandas as pd

for d in (MUSIC_DIR, HTML_DIR, SPLIT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Folder ready: {Path(d).resolve()}")

# ── hash cache ─────────────────────────────────────────────────────────────
cache_path = Path(HASH_CACHE_FILE)
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        hash_cache: dict = json.load(f)
    print(f"📦 Loaded hash cache ({len(hash_cache)} entries)")
else:
    hash_cache = {}
    print("📦 Fresh hash cache")

def save_cache():
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(hash_cache, f, indent=2)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
    print("✅ ffmpeg found")
except FileNotFoundError:
    print("❌ ffmpeg not found — install it before running cells 5 and 6")

## Cell 3 — Helpers

In [ ]:
# ── network ────────────────────────────────────────────────────────────────

def fetch_html(url: str, retries: int = MAX_RETRIES) -> str | None:
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as e:
            if e.code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({e.code}), waiting {wait}s (attempt {attempt}/{retries})...")
                time.sleep(wait)
            else:
                print(f"    ❌ HTTP {e.code} — {url}")
                return None
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Network error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    print(f"    ❌ Gave up after {retries} attempts: {url}")
    return None


def extract_pronunc_hash(html: str) -> str | None:
    m = re.search(r'data-pronunc="([^"]+)"', html)
    return m.group(1) if m else None


def audio_url(skin_hash: str) -> str:
    return (
        "https://www.blueletterbible.org/lang/lexicon/"
        f"lexPronouncePlayer.cfm?skin={skin_hash}"
    )

import requests
def download_audio(url: str, dest_dir: Path, strong_key: str,
                   retries: int = MAX_RETRIES) -> Path | None:
    session = requests.Session()
    session.headers.update(HEADERS)
    
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=30, stream=True)
            
            if resp.status_code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({resp.status_code}), waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            
            # get filename from Content-Disposition or fallback
            cd = resp.headers.get("Content-Disposition", "")
            fname_match = re.search(
                r'filename\*?=(?:UTF-8\'\'|"?)([^";\r\n]+)', cd, re.I
            )
            if fname_match:
                # server sends full server-side path like /mnt/nginx-server/.../h0001.mp3
                # take only the basename, just like curl -J does
                filename = Path(urllib.parse.unquote(fname_match.group(1).strip('"'))).name
            else:
                ct = resp.headers.get("Content-Type", "")
                ext = ".mp3" if "mpeg" in ct else ".ogg" if "ogg" in ct else ".audio"
                filename = f"{strong_key}{ext}"
            
            dest = dest_dir / filename
            with open(dest, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return dest
            
        except requests.exceptions.HTTPError as e:
            print(f"    ❌ HTTP error: {e}")
            return None
        except requests.exceptions.RequestException as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Request error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    
    print(f"    ❌ Gave up after {retries} attempts: {strong_key}")
    return None

# ── ffmpeg helpers ─────────────────────────────────────────────────────────
def detect_silences(audio_path: Path,
                    noise_db: int = SILENCE_DB,
                    min_dur: float = SILENCE_MIN_DUR,
                    total_dur: float = None,
                   spikes_merge_max: float = SILENCE_SPIKES_MERGE_DURATION_MAX) -> list[dict]:
    cmd = [
        "ffmpeg", "-i", str(audio_path),
        "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
        "-f", "null", "-"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stderr
    starts = re.findall(r"silence_start:\s*([\d.]+)", output)
    ends   = re.findall(r"silence_end:\s*([\d.]+)",   output)
    durs   = re.findall(r"silence_duration:\s*([\d.]+)", output)
    silences = [
        {"silence_start": float(s), "silence_end": float(e), "silence_dur": float(d)}
        for s, e, d in zip(starts, ends, durs)
    ]

    # merge gaps split by short noise spikes
    merged = []
    for s in silences:
        if merged and (s["silence_start"] - merged[-1]["silence_end"]) <= spikes_merge_max:
            merged[-1]["silence_end"] = s["silence_end"]
            merged[-1]["silence_dur"] = merged[-1]["silence_end"] - merged[-1]["silence_start"]
        else:
            merged.append(s.copy())

    # drop trailing silence at end of file
    if total_dur is not None:
        merged = [s for s in merged if s["silence_end"] < total_dur - SILENCE_TRAILING_SILENCE_MAX_DURATION_TO_BE_MERGED_WITH_PREVIOUS_CHUNK]

    return merged


def get_audio_duration(audio_path: Path) -> float | None:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return None


print("✅ Helpers ready")

# Action pipelines

## Cell 4 — Download loop (resume-safe)

Re-run freely — cached HTML and existing audio files are skipped.

In [ ]:
# will redownload audio files! (As there is no hash decoder that would allow to see which files are here..)
skipped_no_hash = []
skipped_dl_fail = []
downloaded      = []
already_had     = []

VERBOSE = False
html_root  = Path(HTML_DIR)
music_root = Path(MUSIC_DIR)
START=2501
END=8674#2500
from tqdm.notebook import tqdm
for num in tqdm(range(START, END + 1)):
    key     = f"h{num}"
    html_fp = html_root / f"{key}.html"

    # ── STEP 1: get pronunc hash ───────────────────────────────────────────
    # Priority: in-memory cache → saved HTML on disk → live fetch
    if key in hash_cache:
        skin = hash_cache[key]
    elif html_fp.exists():
        with open(html_fp, "r", encoding="utf-8") as f:
            cached_html = f.read()
        skin = extract_pronunc_hash(cached_html)
        if(VERBOSE):
            print(f"[{num}/{END}] 📄 {key}: hash from saved HTML")
        hash_cache[key] = skin
        save_cache()
    else:
        lex_url = f"https://www.blueletterbible.org/lexicon/{key}/kjv/wlc/0-1/"
        if(VERBOSE):
            print(f"[{num}/{END}] 🌐 {key}: fetching {lex_url}")
        html = fetch_html(lex_url)
        if html is None:
            print(f"  ⚠️  Could not fetch page for {key}, skipping")
            skipped_dl_fail.append(key)
            continue
        # save HTML to disk
        with open(html_fp, "w", encoding="utf-8") as f:
            f.write(html)
        skin = extract_pronunc_hash(html)
        hash_cache[key] = skin
        save_cache()
        time.sleep(DELAY_OK)

    if not skin:
        print(f"[{num}/{END}] ⚪ {key}: no data-pronunc found — no audio")
        skipped_no_hash.append(key)
        continue

    # ── STEP 2: skip if any audio file for this key already exists ─────────
    existing = list(music_root.glob(f"{key}.*"))
    if existing:
        if(VERBOSE):
            print(f"[{num}/{END}] ✔️  {key}: already have {existing[0].name}")
        already_had.append(key)
        continue

    # ── STEP 3: download audio ─────────────────────────────────────────────
    aurl = audio_url(skin)
    if(VERBOSE):
        print(f"[{num}/{END}] 🎵 {key}: downloading...")
    dest = download_audio(aurl, music_root, key)
    if dest:
        if(VERBOSE):
            print(f"  ✅ Saved → {dest.name}")
        downloaded.append(str(dest))
    else:
        skipped_dl_fail.append(key)

    time.sleep(DELAY_OK)

print()
print("=" * 50)
print(f"✅ Newly downloaded : {len(downloaded)}")
print(f"✔️  Already had      : {len(already_had)}")
print(f"⚪ No audio         : {len(skipped_no_hash)}  {skipped_no_hash[:10]}")
print(f"❌ Failed           : {len(skipped_dl_fail)}  {skipped_dl_fail}")

## Cell 5 — Silence analysis → CSV

Runs `ffmpeg silencedetect` on every file in `music/`.  
**One row per silence gap** — so a file with 3 gaps produces 3 rows (plus metadata columns shared across all rows for that file).  
Re-running skips already-analysed keys.

### Inferred audio structure
```
Single-entry, one pronunciation variant:
  [H-number spoken]  GAP-0(long)  [pronunc]  (end)

Single-entry, two pronunciation variants:
  [H-number spoken]  GAP-0(long)  [pronunc A]  GAP-1(short)  [pronunc B]  (end)

Two lexicon entries:
  [H-number spoken]  GAP-0(long)  [pronunc 1A]  GAP-1(short?)  [pronunc 1B?]
  GAP-2(long)  ["second entry"]  GAP-3(long)  [pronunc 2A]  GAP-4(short?)  [pronunc 2B?]  (end)
```
Gap index (`silence_index`, 0-based) tells you position in the file.  
Longer gaps mark structural boundaries; shorter gaps separate pronunciation variants.

In [ ]:
csv_path   = Path(SILENCE_CSV_FILE)
music_root = Path(MUSIC_DIR)
LIMIT_UPTO_ANALYZE_COUNT = -1#5500#1000
OFFSET_FROM_ANALYZE_COUNT = -1#499
PRINT_VERBOSE_LOG=False #True
PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG=False
#noise_db
        # "noise_db", SILENCE_DB),
        # "min_dur", SILENCE_MIN_DUR),
        # "spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
SILENCE_EXCEPTIONS = {
    "h0067": {"min_dur": 0.45},           # intra-word pause filtered out
    "h0298": {"min_dur": 0.30, "noise_db": -35}, #shorter pause before geseiahas lexicon
    "h0188": {"spikes_merge_max": 0.15},  # was merging across short speech
    "h0411": {"noise_db": -32},
    "h0415": {"noise_db": -32,
         "min_dur": 0.19},
    "h0369": {"noise_db": -40,
         "min_dur": 0.30},
    # add more as needed...
}

    
for t in ["h0439", "h0413", "h0452", "h0453", "h0454", "h0500", "h0373"]:
    SILENCE_EXCEPTIONS[t] = {"noise_db": -32}
for t1 in ["h0535", "h0536", "h0616", "h0741", "h0764", "h0796", "h0797", "h0832", "h0845", "h0848", "h0875", "h0876", "h0501", "h0526", "h0533", "h0534", "h0537", "h0538", "h0539", "h0542", "h0544", "h0545", "h0553", "h0555", "h0557", "h0558", "h0559", "h0560", "h0561", "h0562", "h0564", "h0566", "h0569", "h0570", "h0571", "h0572", "h0573", "h0574", "h0575", "h0578", "h0579", "h0580", "h0581", "h0582", "h0583", "h0584", "h0585", "h0587", "h0588", "h0589", "h0590", "h0592", "h0593", "h0594", "h0595", "h0596", "h0597", "h0598", "h0599", "h0600", "h0602", "h0603", "h0605", "h0606", "h0607", "h0608", "h0609", "h0612", "h0613", "h0615", "h0617", "h0618", "h0619", "h0620", "h0621", "h0622", "h0623", "h0625", "h0626", "h0627", "h0628", "h0629", "h0630", "h0631", "h0633", "h0634", "h0635", "h0636", "h0637", "h0638", "h0639", "h0640", "h0642", "h0643", "h0644", "h0645", "h0646", "h0647", "h0648", "h0650", "h0651", "h0652", "h0653", "h0654", "h0655", "h0656", "h0657", "h0658", "h0659", "h0660", "h0661", "h0662", "h0663", "h0664", "h0665", "h0666", "h0667", "h0668", "h0669", "h0670", "h0673", "h0674", "h0675", "h0676", "h0677", "h0678", "h0679", "h0680", "h0681", "h0682", "h0683", "h0684", "h0685", "h0686", "h0687", "h0689", "h0690", "h0693", "h0694", "h0695", "h0696", "h0697", "h0698", "h0699", "h0700", "h0701", "h0702", "h0705", "h0706", "h0707", "h0708", "h0709", "h0710", "h0711", "h0712", "h0713", "h0714", "h0716", "h0717", "h0719", "h0720", "h0722", "h0725", "h0727", "h0728", "h0729", "h0730", "h0731", "h0732", "h0733", "h0734", "h0735", "h0736", "h0737", "h0738", "h0739", "h0740", "h0742", "h0743", "h0744", "h0746", "h0747", "h0748", "h0749", "h0750", "h0751", "h0752", "h0753", "h0754", "h0755", "h0756", "h0757", "h0759", "h0761", "h0762", "h0765", "h0766", "h0768", "h0769", "h0770", "h0771", "h0772", "h0773", "h0774", "h0775", "h0776", "h0777", "h0778", "h0779", "h0780", "h0781", "h0782", "h0783", "h0784", "h0785", "h0786", "h0787", "h0788", "h0790", "h0791", "h0792", "h0793", "h0794", "h0795", "h0798", "h0799", "h0800", "h0801", "h0802", "h0803", "h0804", "h0805", "h0806", "h0807", "h0808", "h0809", "h0810", "h0811", "h0812", "h0813", "h0814", "h0815", "h0817", "h0818", "h0819", "h0820", "h0822", "h0823", "h0825", "h0827", "h0828", "h0829", "h0830", "h0831", "h0834", "h0835", "h0836", "h0837", "h0840", "h0841", "h0842", "h0843", "h0844", "h0849", "h0850", "h0852", "h0853", "h0854", "h0855", "h0856", "h0857", "h0858", "h0859", "h0860", "h0861", "h0862", "h0864", "h0866", "h0867", "h0868", "h0869", "h0870", "h0871", "h0872", "h0873", "h0874", "h0877", "h0878", "h0879", "h0880", "h0881", "h0882", "h0883", "h0884", "h0885", "h0886", "h0887", "h0888", "h0889", "h0890", "h0891", "h0892", "h0893", "h0894", "h0895", "h0896", "h0897", "h0898", "h0899", "h0901", "h0902", "h0903", "h0904", "h0905", "h0906", "h0907", "h0908", "h0909", "h0910", "h0912", "h0913", "h0914", "h0915", "h0917", "h0918", "h0919", "h0920", "h0922", "h0923", "h0924", "h0925", "h0927", "h0928", "h0929", "h0930", "h0931", "h0932", "h0933", "h0934", "h0935", "h0936", "h0937", "h0938", "h0939", "h0941", "h0942", "h0943", "h0944", "h0945", "h0946", "h0947", "h0948", "h0949", "h0950", "h0951", "h0952", "h0953", "h0954", "h0955", "h0956", "h0957", "h0958", "h0959", "h0960", "h0961", "h0963", "h0964", "h0965", "h0966", "h0967", "h0968", "h0970", "h0971", "h0972", "h0973", "h0974", "h0975", "h0976", "h0977", "h0978", "h0980", "h0982", "h0983", "h0984", "h0985", "h0986", "h0987", "h0988", "h0989", "h0990", "h0991", "h0992", "h0993", "h0994", "h0995", "h0996", "h0997", "h0998", "h0999", "h0502", "h0503", "h0506", "h0550", "h0632", "h0763", "h0833", "h0926", "h0979", "h0565", "h0568", "h0577", "h0586", "h0611", "h0624", "h0641", "h0672", "h0723", "h0724", "h0821", "h0851", "h0981"]:
    SILENCE_EXCEPTIONS[t1] = {"noise_db": -32}
for t2 in ["h0501", "h0526", "h0539", "h0570", "h0571", "h0589", "h0626", "h0629", "h0630", "h0648", "h0652", "h0673", "h0754", "h0755", "h0769", "h0773", "h0774", "h0787", "h0795", "h0805", "h0859", "h0871", "h0873", "h0901", "h0923", "h0929", "h0939", "h0945", "h0958", "h0975", "h0996"]:
    SILENCE_EXCEPTIONS[t2] = {"noise_db": -32,
                             "min_dur": 0.19}
for tsp in ["h0526", "h0570", "h0629"]:
    SILENCE_EXCEPTIONS[tsp] = {"noise_db": -30,
                             "min_dur": 0.12}
# load existing CSV to skip already-analysed keys
if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    already_analysed = set(df_existing["key"].unique())
    print(f"📊 Loaded existing CSV ({len(df_existing)} rows, {len(already_analysed)} keys)")
else:
    df_existing = pd.DataFrame()
    already_analysed = set()
    print("📊 No existing CSV — starting fresh")

new_rows = []

# collect all audio files, sorted numerically
def sort_key(p):
    m = re.search(r"\d+", p.stem)
    return int(m.group()) if m else 0
    
audio_files = []
for ext in ("mp3", "ogg", "audio"):
    audio_files += sorted(music_root.glob(f"h*.{ext}"), key=sort_key)

from tqdm.notebook import tqdm
cnt = 0
for audio_fp in tqdm(audio_files):
    cnt=cnt+1
    if(LIMIT_UPTO_ANALYZE_COUNT>0):
        if(cnt>LIMIT_UPTO_ANALYZE_COUNT):
            break
    if(OFFSET_FROM_ANALYZE_COUNT>0):
        if(cnt<OFFSET_FROM_ANALYZE_COUNT):
            continue
    
    m = re.search(r"(h\d+)", audio_fp.stem, re.I)
    if not m:
        print(f"⚠️  Can't parse key from: {audio_fp.name} — skipping")
        continue
    key = m.group(1).lower()

    if key in already_analysed:
        if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
            print(f"✔️  {key}: already analysed")
        continue

    if(PRINT_VERBOSE_LOG):
        print(f"🔍 {key}: analysing {audio_fp.name}...", end=" ", flush=True)
        print(SILENCE_EXCEPTIONS.get(key, {}))
    total_dur = get_audio_duration(audio_fp)
    overrides = SILENCE_EXCEPTIONS.get(key, {})
    silences = detect_silences(
        audio_fp,
        total_dur=total_dur,
        noise_db=overrides.get("noise_db", SILENCE_DB),
        min_dur=overrides.get("min_dur", SILENCE_MIN_DUR),
        spikes_merge_max=overrides.get("spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
    )
    if(len(silences)<3):
        if(PRINT_VERBOSE_LOG):
            print(f"RETRY!: {len(silences)} gap(s)  (total {total_dur:.2f}s)")
        if(PRINT_VERBOSE_LOG):
            print(f"🔍 {key}: analysing {audio_fp.name}...", end=" ", flush=True)
            print(SILENCE_EXCEPTIONS.get(key, {}))
        total_dur = get_audio_duration(audio_fp)
        overrides = SILENCE_EXCEPTIONS.get(key, {})
        silences = detect_silences(
            audio_fp,
            total_dur=total_dur,
            noise_db=overrides.get("noise_db", -30),
            min_dur=overrides.get("min_dur", 0.12),
            spikes_merge_max=overrides.get("spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
        )
    if(PRINT_VERBOSE_LOG):
        print(f"{len(silences)} gap(s)  (total {total_dur:.2f}s)")

    base_row = {
        "key":           key,
        "filename":      audio_fp.name,
        "total_dur":     total_dur,
        "silence_count": len(silences),
    }

    if not silences:
        # one row with nulls for silence columns
        new_rows.append({**base_row,
                         "silence_index": None,
                         "silence_start": None,
                         "silence_end":   None,
                         "silence_dur":   None})
    else:
        for idx, s in enumerate(silences):
            new_rows.append({**base_row,
                             "silence_index": idx,
                             "silence_start": s["silence_start"],
                             "silence_end":   s["silence_end"],
                             "silence_dur":   s["silence_dur"]})

# save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all.to_csv(csv_path, index=False)
    print(f"\n💾 Saved {len(df_all)} total rows → {csv_path}")
else:
    df_all = df_existing
    print("\nNo new files to analyse.")

# summary table
if not df_all.empty:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

In [ ]:
invalid_keys = (
    df_all.groupby("key")["silence_count"]
    .first()
    .pipe(lambda s: s[(s % 3 != 0) | (s == 0)])
    .index
)
df_all = df_all[~df_all["key"].isin(invalid_keys)]
df_all.to_csv(csv_path, index=False)
df_all = pd.read_csv(csv_path)

In [ ]:
df = pd.read_csv(csv_path)
#df = df[df["key"] != "h0411"]
#df = df[~df["key"].isin(["h0535", "h0536", "h0616", "h0741", "h0764", "h0796", "h0797", "h0832", "h0845", "h0848", "h0875", "h0876", "h0501", "h0526", "h0533", "h0534", "h0537", "h0538", "h0539", "h0542", "h0544", "h0545", "h0553", "h0555", "h0557", "h0558", "h0559", "h0560", "h0561", "h0562", "h0564", "h0566", "h0569", "h0570", "h0571", "h0572", "h0573", "h0574", "h0575", "h0578", "h0579", "h0580", "h0581", "h0582", "h0583", "h0584", "h0585", "h0587", "h0588", "h0589", "h0590", "h0592", "h0593", "h0594", "h0595", "h0596", "h0597", "h0598", "h0599", "h0600", "h0602", "h0603", "h0605", "h0606", "h0607", "h0608", "h0609", "h0612", "h0613", "h0615", "h0617", "h0618", "h0619", "h0620", "h0621", "h0622", "h0623", "h0625", "h0626", "h0627", "h0628", "h0629", "h0630", "h0631", "h0633", "h0634", "h0635", "h0636", "h0637", "h0638", "h0639", "h0640", "h0642", "h0643", "h0644", "h0645", "h0646", "h0647", "h0648", "h0650", "h0651", "h0652", "h0653", "h0654", "h0655", "h0656", "h0657", "h0658", "h0659", "h0660", "h0661", "h0662", "h0663", "h0664", "h0665", "h0666", "h0667", "h0668", "h0669", "h0670", "h0673", "h0674", "h0675", "h0676", "h0677", "h0678", "h0679", "h0680", "h0681", "h0682", "h0683", "h0684", "h0685", "h0686", "h0687", "h0689", "h0690", "h0693", "h0694", "h0695", "h0696", "h0697", "h0698", "h0699", "h0700", "h0701", "h0702", "h0705", "h0706", "h0707", "h0708", "h0709", "h0710", "h0711", "h0712", "h0713", "h0714", "h0716", "h0717", "h0719", "h0720", "h0722", "h0725", "h0727", "h0728", "h0729", "h0730", "h0731", "h0732", "h0733", "h0734", "h0735", "h0736", "h0737", "h0738", "h0739", "h0740", "h0742", "h0743", "h0744", "h0746", "h0747", "h0748", "h0749", "h0750", "h0751", "h0752", "h0753", "h0754", "h0755", "h0756", "h0757", "h0759", "h0761", "h0762", "h0765", "h0766", "h0768", "h0769", "h0770", "h0771", "h0772", "h0773", "h0774", "h0775", "h0776", "h0777", "h0778", "h0779", "h0780", "h0781", "h0782", "h0783", "h0784", "h0785", "h0786", "h0787", "h0788", "h0790", "h0791", "h0792", "h0793", "h0794", "h0795", "h0798", "h0799", "h0800", "h0801", "h0802", "h0803", "h0804", "h0805", "h0806", "h0807", "h0808", "h0809", "h0810", "h0811", "h0812", "h0813", "h0814", "h0815", "h0817", "h0818", "h0819", "h0820", "h0822", "h0823", "h0825", "h0827", "h0828", "h0829", "h0830", "h0831", "h0834", "h0835", "h0836", "h0837", "h0840", "h0841", "h0842", "h0843", "h0844", "h0849", "h0850", "h0852", "h0853", "h0854", "h0855", "h0856", "h0857", "h0858", "h0859", "h0860", "h0861", "h0862", "h0864", "h0866", "h0867", "h0868", "h0869", "h0870", "h0871", "h0872", "h0873", "h0874", "h0877", "h0878", "h0879", "h0880", "h0881", "h0882", "h0883", "h0884", "h0885", "h0886", "h0887", "h0888", "h0889", "h0890", "h0891", "h0892", "h0893", "h0894", "h0895", "h0896", "h0897", "h0898", "h0899", "h0901", "h0902", "h0903", "h0904", "h0905", "h0906", "h0907", "h0908", "h0909", "h0910", "h0912", "h0913", "h0914", "h0915", "h0917", "h0918", "h0919", "h0920", "h0922", "h0923", "h0924", "h0925", "h0927", "h0928", "h0929", "h0930", "h0931", "h0932", "h0933", "h0934", "h0935", "h0936", "h0937", "h0938", "h0939", "h0941", "h0942", "h0943", "h0944", "h0945", "h0946", "h0947", "h0948", "h0949", "h0950", "h0951", "h0952", "h0953", "h0954", "h0955", "h0956", "h0957", "h0958", "h0959", "h0960", "h0961", "h0963", "h0964", "h0965", "h0966", "h0967", "h0968", "h0970", "h0971", "h0972", "h0973", "h0974", "h0975", "h0976", "h0977", "h0978", "h0980", "h0982", "h0983", "h0984", "h0985", "h0986", "h0987", "h0988", "h0989", "h0990", "h0991", "h0992", "h0993", "h0994", "h0995", "h0996", "h0997", "h0998", "h0999", "h0502", "h0503", "h0506", "h0550", "h0632", "h0763", "h0833", "h0926", "h0979", "h0565", "h0568", "h0577", "h0586", "h0611", "h0624", "h0641", "h0672", "h0723", "h0724", "h0821", "h0851", "h0981"])]
filt=["h0501", "h0526", "h0539", "h0570", "h0571", "h0589", "h0626", "h0629", "h0630", "h0648", "h0652", "h0673", "h0754", "h0755", "h0769", "h0773", "h0774", "h0787", "h0795", "h0805", "h0859", "h0871", "h0873", "h0901", "h0923", "h0929", "h0939", "h0945", "h0958", "h0975", "h0996"]
df = df[~df["key"].isin(filt)]
df.to_csv(csv_path, index=False)
df = pd.read_csv(csv_path)

In [ ]:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

In [ ]:
df_all = pd.read_csv("blb_silence_analysis.csv")
gapr = set(range(0,10))
divr = set(i for i in range (0,10) if (i%3 ==0 and i!=0) )
for n_gaps in (gapr-divr):
    keys = df_all[df_all["silence_count"] == n_gaps]["key"].unique()
    for key in keys:
        rows = df_all[df_all["key"] == key].dropna(subset=["silence_start"])
        print(f"\n{'~'*50}")
        if(n_gaps==0):
            print(f"{key} ({n_gaps} gaps)")
            continue
        print(f"{key} ({n_gaps} gaps), total_dur={rows.iloc[0]['total_dur']:.2f}s")
        for _, r in rows.iterrows():
            print(f"  gap {int(r['silence_index'])}: {r['silence_start']:.2f}→{r['silence_end']:.2f}  (dur={r['silence_dur']:.2f}s)")

# and a couple of the 6-gap ones to confirm they are genuine 2-entry files
ppp="""
print("\n=== sample of 6-gap files ===")
keys_6 = df_all[df_all["silence_count"] == 6]["key"].unique()[:3]
for key in keys_6:
    rows = df_all[df_all["key"] == key].dropna(subset=["silence_start"])
    print(f"\n{key} (6 gaps), total_dur={rows.iloc[0]['total_dur']:.2f}s")
    for _, r in rows.iterrows():
        print(f"  gap {int(r['silence_index'])}: {r['silence_start']:.2f}→{r['silence_end']:.2f}  (dur={r['silence_dur']:.2f}s)")
"""

In [ ]:
dfdelstr="["
for n_gaps in ([21]):
    keys = df_all[df_all["silence_count"] == n_gaps]["key"].unique()
    for key in keys:
        dfdelstr= dfdelstr+ f'"{key}", '
dfdelstr=dfdelstr[:-2]
dfdelstr=dfdelstr+']'

print(dfdelstr)

In [ ]:
key = "h0503"
SILENCE_EXCEPTIONS[key]={"noise_db": -30,
                             "min_dur": 0.12}
audio_fp= Path(music_root) / (key +".mp3")
total_dur = get_audio_duration(audio_fp)
overrides = SILENCE_EXCEPTIONS.get(key, {})
silences = detect_silences(
        audio_fp,
        total_dur=total_dur,
        noise_db=overrides.get("noise_db", SILENCE_DB),
        min_dur=overrides.get("min_dur", SILENCE_MIN_DUR),
        spikes_merge_max=overrides.get("spikes_merge_max", SILENCE_SPIKES_MERGE_DURATION_MAX),
    )
print(silences)
print(total_dur)
print(len(silences))

In [ ]:
df_all = pd.read_csv("blb_silence_analysis.csv")
gapr = set(range(0,10))
divr = set(i for i in range (0,10) if (i%3 ==0 and i!=0) )
dfdelstr='df[~df["key"].isin(['
for n_gaps in (gapr-divr):
    keys = df_all[df_all["silence_count"] == n_gaps]["key"].unique()
    for key in keys:
        dfdelstr= dfdelstr+ f'"{key}", '
dfdelstr=dfdelstr[:-2]
dfdelstr=dfdelstr+'])]'

print(dfdelstr)

In [ ]:
df = pd.read_csv("blb_silence_analysis.csv")
df = df[df["key"] != "h0067"]
df.to_csv("blb_silence_analysis.csv", index=False)

## Cell 6 — Split & reformat with ffmpeg

### Segment assignment logic

```
speech_seg[0]           → ALWAYS DROP  (speaker says the Strong's number)
speech_seg[1]           → entry-1, pronunciation A          → h<N>.mp3
speech_seg[2] (if any)  → depends on gap before it:
    SHORT gap            → entry-1, pronunciation B (concat with A) → h<N>.mp3
    LONG gap             → entry-2 preamble ("second entry") → DROP
speech_seg[3] (if any)  → entry-2, pronunciation A           → h<N>-2.mp3
...and so on.
```

**Short vs long** gap heuristic: threshold = 70% of median gap duration per file.  
Gaps >= threshold are "long" (structural boundary); below are "short" (variant separator).

Output in `strongs_p/`:
- `h1.mp3`   — entry 1 (all pronunciation variants concatenated if >1)
- `h1-2.mp3` — entry 2 (if a second lexicon entry exists)

Re-running skips already-split keys.

In [ ]:
import statistics

split_root = Path(SPLIT_DIR)
music_root = Path(MUSIC_DIR)
csv_path   = Path(SILENCE_CSV_FILE)

if not csv_path.exists():
    raise RuntimeError("Run Cell 5 first to generate the silence CSV.")

df_all = pd.read_csv(csv_path)


# ── ffmpeg extract/concat helpers ──────────────────────────────────────────

def ffmpeg_extract(src: Path, start: float, end: float, dest: Path):
    """Cut [start, end) seconds from src → dest (mp3)."""
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),
        "-ss", f"{start:.6f}",
        "-t",  f"{end - start:.6f}",
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)


def ffmpeg_concat(parts: list[Path], dest: Path):
    """Concatenate audio segments into dest."""
    if len(parts) == 1:
        shutil.copy(parts[0], dest)
        return
    list_file = dest.parent / f"_cl_{dest.stem}.txt"
    with open(list_file, "w") as f:
        for p in parts:
            f.write(f"file '{p.resolve()}'\n")
    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(list_file),
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    list_file.unlink(missing_ok=True)


# ── per-key splitting ──────────────────────────────────────────────────────
from tqdm.notebook import tqdm
def split_key(key: str, df: pd.DataFrame) -> list[Path]:
    rows = df[df["key"] == key].sort_values("silence_index")
    if rows.empty:
        return []

    src_name = rows.iloc[0]["filename"]
    src      = music_root / src_name
    if not src.exists():
        print(f"  ⚠️  Source not found: {src}")
        return []

    total_dur  = float(rows.iloc[0]["total_dur"])
    n_silences = int(rows.iloc[0]["silence_count"])

    if n_silences == 0:
        out = split_root / f"{key}.mp3"
        shutil.copy(src, out)
        return [out]

    sil_rows = rows.dropna(subset=["silence_start"]).sort_values("silence_index")
    silences = list(zip(
        sil_rows["silence_start"].astype(float),
        sil_rows["silence_end"].astype(float),
        sil_rows["silence_dur"].astype(float),
    ))

    # number of entries = gap_count / 3  (always divisible by 3)
    n_entries = n_silences // 3

    # for entry N (0-based):
    #   content starts at: end of gap (N*3 + 1)
    #   content ends at:   start of gap (N*3 + 2)  ... which gives [pronunc → gap → repeat]
    #   gap (N*3 + 2) end leads into next entry's announcement (or end of file)
    # but we want to INCLUDE gap-2 (the silence between pronunc and repeat) in the output
    # so: start = end of gap(N*3+1), end = start of gap(N*3+3) or total_dur


    saved = []
    for n in tqdm(range(n_entries), leave=False):
        i_start = n * 3 + 1
        i_end   = n * 3 + 3
        content_start = silences[i_start][1]                          # end of gap
        content_end   = silences[i_end][0] if i_end < len(silences) else total_dur  # start of gap or EOF
    
        out_name = f"{key}.mp3" if n == 0 else f"{key}-{n + 1}.mp3"
        out_path = split_root / out_name
        ffmpeg_extract(src, content_start, content_end, out_path)
        saved.append(out_path)
    return saved


# ── run over all analysed keys ─────────────────────────────────────────────

valid_keys = (
    df_all.groupby("key")["silence_count"]
    .first()
    .pipe(lambda s: s[(s % 3 == 0) & (s != 0)])
    .index
)

keys_in_csv = sorted(
    valid_keys,
    key=lambda k: int(re.search(r"\d+", k).group())
)
from tqdm.notebook import tqdm
for key in tqdm(keys_in_csv):
    if (split_root / f"{key}.mp3").exists():
        if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
            print(f"✔️  {key}: already split")
        continue

    try:
        saved = split_key(key, df_all)
        if saved:
            if(PRINT_VERBOSE_VERBOSE_ALREADY_ANALYSED_LOG):
                print(f"✅ {key}: → {', '.join(p.name for p in saved)}")
        else:
            print(f"⚠️  {key}: nothing produced")
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr[-300:] if e.stderr else ""
        print(f"❌ {key}: ffmpeg error — {stderr_tail}")
    except Exception as e:
        print(f"❌ {key}: {e}")

print("\n🏁 Done.")